# Drought Metrics

This notebook calculates two drought metrics, the Palmer Drought Severity Index (PDSI) and the Evaporative Demand Drought Index (EDDI), using WRF data in the AE catalog. PDSI relies on Potential Evapotranspiration (PET), which is computed using the Penman-Monteith method. At the end of the notebook, the user will be able to export monthly PDSI and EDDI under different global warming levels for a specific lat/lon as netcdf files for further analysis.

**Intended Application:** As a user, I want to <font color="red"> export future drought indices for different global warming levels </font> by:
1. Calculating PET using the Penman-Monteith Method
2. Calculating the PDSI using PET and exporting the monthly timeseries to a netcdf
3. Calculating the EDDI and exporting the monthly timeseries to a netcdf

**Runtime:** With the default settings, this notebook takes approximately **10 minutes** to run from start to finish. Modifications to selections may increase the runtime.

## Step 0: Setup

In [ ]:
# Install climate_indices
!pip install -qq git+https://github.com/monocongo/climate_indices.git

In [ ]:
import climakitae as ck
import xclim
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar
from climate_indices import palmer
from xclim.indices.stats import standardized_index
from climakitae.core.data_export import export

In [ ]:
# Define the lat/lon for the location of interest and the variables needed for the Penman-Monteith PET method
LAT = 37.787964
LON = -122.065063
WARMING_LEVELS = [0.8, 2.0]

## Step 1: Fetch Data and Calculate PET

### Penman-Monteith PET Method (most physically consistent)
We will be using the PET method for calculating PDSI, since it is the most physically consistent method. Below, we list out the variables we will need.

**Variables needed:**
- `t2` (hourly) → daily `tasmin` and `tasmax` derived via resample
- `relative humidity`
- `radiation flux`
    - rsds (daily)
    - rsus (hourly → daily mean)
    - rlds (daily)
    - rlus (hourly → daily mean)
- `wind speed (10m wind will be converted to 2m)`

In [ ]:
# Initialize the ClimateData object and define the processors needed.
cd = ck.ClimateData(verbosity=-2)
processes = {
    "clip": (LAT, LON),
    "warming_level": {
        "warming_levels": WARMING_LEVELS,
        "add_dummy_time": True
    },
}

def rename_sims_to_gcm(ds):
    """"Renames the 'sim' coordinate in the dataset to only contain the GCM name for easier grouping later."""
    ds = ds.copy()
    new_sims = [s.split('_')[2] for s in ds['sim'].values]
    ds['sim'] = ('sim', new_sims)
    return ds

def fetch(variable, table_id="day", units=None):
    """Fetch a WRF variable from the AE catalog with the configured processes."""
    proc = {**processes}
    if units is not None:
        proc["convert_units"] = units
    return (
        cd.catalog("cadcat")
        .activity_id("WRF")
        .institution_id("UCLA")
        .table_id(table_id)
        .grid_label("d03")
        .variable(variable)
        .processes(proc)
        .get()
    )

def get_and_transform(variable, table_id="day", units=None, transform=None):
    """Fetching variables and applying a transformation if it's passed in."""
    print(f"\nFetching {variable}...")
    da = fetch(variable, table_id=table_id, units=units)
    da = da.unify_chunks()  # Ensure the data is chunked for Dask processing
    if transform:
        da = transform(da)
    # Drop leap days
    da = da.sel(time=~((da['time.month'] == 2) & (da['time.day'] == 29)))
    # Rename sims to only contain the GCM name for easier grouping later
    da = rename_sims_to_gcm(da)
    return da

# Daily variables
rh         = get_and_transform("rh")
sw_dwn     = get_and_transform("sw_dwn")
lw_dwn     = get_and_transform("lw_dwn")
wspd10mean = get_and_transform("wspd10mean")
prec       = get_and_transform("prec")
tasmin     = get_and_transform("t2min")
tasmax     = get_and_transform("t2max")
rsus_daily = get_and_transform("swupb", table_id="1hr", transform=lambda da: da.resample(time="1D").mean())
rlus_daily = get_and_transform("lwupb",  table_id="1hr", transform=lambda da: da.resample(time="1D").mean())

# xclim requires relative humidity as a fraction (0–1), not a percentage
hurs_frac  = (rh / 100)
hurs_frac.rh.attrs['units']  = '1' # Update the units to reflect the transformation to a fraction

# Ensure all radiation flux variables carry the same units
sw_dwn.sw_dwn.attrs['units'] = 'W m-2'
lw_dwn.lw_dwn.attrs['units'] = 'W m-2'
rsus_daily.swupb.attrs['units'] = 'W m-2'
rlus_daily.lwupb.attrs['units'] = 'W m-2'

print("Finished.")

#### Calculate PET from `xclim` using the Penman-Monteith Method

In [ ]:
# Call xclim.indicies.potential_evapotranspiration on the entire dataset as it is
pet_pm = xclim.indices.potential_evapotranspiration(
    tasmin=tasmin.t2min,
    tasmax=tasmax.t2max,
    hurs=hurs_frac.rh,
    rsds=sw_dwn.sw_dwn,
    rsus=rsus_daily.swupb,
    rlds=lw_dwn.lw_dwn,
    rlus=rlus_daily.lwupb,
    sfcWind=wspd10mean.wspd10mean,
    method="FAO_PM98",
)

## Step 2: Calculate drought indices

#### Helper Function

In [ ]:
### Combining WL objects together, historical WL as 2000-2030, future WL as 2030-2060
def combine_wl_to_dummy_time(
    da: xr.DataArray,
    baseline_wl: float,
    future_wls: list[float],
    start_date: str = "2000-01-31",
) -> xr.DataArray:
    """
    Combine baseline warming level with multiple future warming levels into one
    DataArray along a new 'combined_wl' dimension.

    Parameters
    ----------
    da : xr.DataArray
        Original data with dims including 'warming_level' and 'time'.
    baseline_wl : float
        The warming level used for the first time segment.
    future_wls : list of float
        Warming levels to concatenate after baseline.
    start_date : str
        Start date for the combined time series (monthly freq).
    
    Returns
    -------
    xr.DataArray
        Combined DataArray with new dimension 'combined_wl' and coordinate labels like "0.8 to 1.5".
    """
    months_per_wl = da.sizes['time']
    total_months = 2 * months_per_wl
    new_time = pd.date_range(start_date, periods=total_months, freq='ME')

    combined_list = []
    combined_labels = []

    for fw in future_wls:
        da_base = da.sel(warming_level=baseline_wl)
        da_future = da.sel(warming_level=fw)

        combined = xr.concat([da_base, da_future], dim='time')
        combined = combined.assign_coords(time=new_time)

        wl_flag = np.array([baseline_wl] * months_per_wl + [fw] * months_per_wl)
        combined = combined.assign_coords(warming_level_flag=('time', wl_flag))

        combined_list.append(combined)
        combined_labels.append(f"{int(baseline_wl * 10):02d}_to_{int(fw * 10):02d}")

    combined_da = xr.concat(combined_list, dim='combined_wl')
    combined_da = combined_da.assign_coords(combined_wl=combined_labels)

    return combined_da

### 2.1. Calculate PDSI

The underlying PDSI computation uses the `climate_indices` library's `palmer_pdsi` function — the same implementation referenced by [drought.gov](https://www.drought.gov). `xr.apply_ufunc` with `vectorize=True` applies it across the `sim` and `combined_wl` dimensions, iterating over each time series independently.

In [ ]:
# Resampling PET and precip to monthly since the function only takes monthly variables
mon_pet    = (pet_pm * 86400 / 25.4).resample(time='1ME').sum()
mon_precip = (prec / 25.4).resample(time='1ME').sum()

In [ ]:
# Creating one Dataset of PET and precip with WLs combined
mon_pet_transform = combine_wl_to_dummy_time(mon_pet, baseline_wl=WARMING_LEVELS[0], future_wls=WARMING_LEVELS[1:])
mon_precip_transform = combine_wl_to_dummy_time(mon_precip.prec, baseline_wl=WARMING_LEVELS[0], future_wls=WARMING_LEVELS[1:])
mon_pet_transform = mon_pet_transform.assign_coords(mon_precip_transform.coords)
combined_ds = xr.Dataset({'precip': mon_precip_transform, 'pet': mon_pet_transform})

In [ ]:
# Load `combined_ds` into memory
with ProgressBar():
    combined_ds = combined_ds.compute()

In [ ]:
def compute_pdsi(precip_1d, pet_1d, awc, data_start_year, calibration_year_initial, calibration_year_final):
    """Compute drought indices for a single 1-D time series of precipitation and PET, and only capture PDSI."""
    pdsi, _, _, _, _ = palmer.pdsi(
        precips=precip_1d,
        pet=pet_1d,
        awc=awc,
        data_start_year=data_start_year,
        calibration_year_initial=calibration_year_initial,
        calibration_year_final=calibration_year_final,
    )
    return pdsi

pdsi_result = xr.apply_ufunc(
    compute_pdsi,
    combined_ds["precip"],        # shape (time, sim)
    combined_ds["pet"],       # shape (time, sim)
    input_core_dims=[["time"], ["time"]],   # operate along time
    output_core_dims=[["time"]],            # output also has time dim
    vectorize=True,                         # loop over non-core dims (sim)
    kwargs={
        "awc": 6,                           # Default Average Water Capacity
        "data_start_year": 2000,
        "calibration_year_initial": 2000,
        "calibration_year_final": 2030,
    }
)

The combined time axis has 720 months total: months 0–359 are the baseline warming level (0.8°C, used for calibration), and months 360–719 are the future warming level period. The export below slices to `time=slice(360, 720)` to retain only the future period.

The exported file will have the following dimensions:
- `time`: 360 months (30-year future warming level period)
- `wl`: warming level pair (e.g., `"08_to_20"` = calibrated on 0.8°C, evaluated on 2.0°C)
- `lat`, `lon`: spatial coordinates (EPSG:4326)
- `sim`: ensemble member

#### Export PDSI

In [ ]:
# Cleaning and labeling the data before exporting it in the next cell
final_pdsi = pdsi_result.isel(time=slice(360, 720))
final_pdsi = final_pdsi.rename({'combined_wl': 'wl'}).rename("pdsi")
final_pdsi = final_pdsi.assign_attrs({
    "long_name": "Palmer Drought Severity Index",
    "units": "from -10 (dry) to +10 (wet)",
})
pdsi_filename = f"pdsi_wl_lat{str(LAT).replace('.', '_')}_lon{str(LON).replace('.', '_')}.nc"

In [ ]:
export(final_pdsi, pdsi_filename, format="NetCDF", mode="local")

### 2.2. Calculate EDDI

EDDI is computed as a standardized anomaly using `xclim`'s `standardized_index`, calibrated over the 0.8°C warming level period (2000–2029), with values clipped to [−2.5, 2.5].

In [ ]:
def compute_eddi_numpy(timeseries_np, time_coords):
    """Wrapper that reconstructs DataArray from numpy for apply_ufunc."""
    ts = xr.DataArray(timeseries_np, coords={"time": time_coords}, dims=["time"])
    eddi = standardized_index(
        da=ts,
        freq='MS',
        window=1,
        dist="gamma",
        method="ML",
        zero_inflated=True,
        fitkwargs={},
        cal_start="2000-01-31",
        cal_end="2029-12-31"
    )
    return eddi.clip(min=-2.5, max=2.5) # Clipping extraneous values to be within the realistic range of EDDI

time_coords = combined_ds.time.values

eddi_result = xr.apply_ufunc(
    compute_eddi_numpy,
    combined_ds['precip'],
    input_core_dims=[['time']],
    output_core_dims=[['time']],
    vectorize=True,
    kwargs={"time_coords": time_coords},
)

#### Export EDDI

The combined time axis has 720 months total: months 0–359 are the baseline warming level (0.8°C, used for calibration), and months 360–719 are the future warming level period. The export below slices to `time=slice(360, 720)` to retain only the future period.

The exported file will have the following dimensions:
- `time`: 360 months (30-year future warming level period)
- `wl`: warming level pair (e.g., `"08_to_20"` = calibrated on 0.8°C, evaluated on 2.0°C)
- `lat`, `lon`: spatial coordinates (EPSG:4326)
- `sim`: ensemble member

In [ ]:
# Saving these results and cleaning the data
final_eddi = eddi_result.isel(time=slice(360, 720))
final_eddi = final_eddi.rename({'combined_wl': 'wl'}).rename("eddi")
final_eddi = final_eddi.assign_attrs({
    "long_name": "Evaporative Demand Drought Index",
    "units": "from -2.5 (wet) to +2.5 (dry)",
})
eddi_filename = f"eddi_wl_lat{str(LAT).replace('.', '_')}_lon{str(LON).replace('.', '_')}.nc"

In [ ]:
export(final_eddi, eddi_filename, format="NetCDF", mode="local")

<br>

## Summary

This notebook demonstrates how you can use `climakitae` variables to compute and export two drought indices for a given location, broken down in the table below:

| Index | Method | Output file |
|-------|--------|-------------|
| PDSI | Palmer–Monteith (`climate_indices`) | `pdsi_wl_*.nc` |
| EDDI | Standardized index (`xclim`) | `eddi_wl_*.nc` |